In [ ]:
%pip install azure-identity azure-ai-evaluation

In [ ]:
import os
from dotenv import load_dotenv
import json
from pathlib import Path

load_dotenv()

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

model_config = {
    "azure_endpoint": ENDPOINT,
    "azure_deployment": DEPLOYMENT_NAME,
    "api_key": API_KEY,
}

current_dir =  "C:/Users/t-toluale/.vscode/Ordering_ChatBot/"
BRAND_CONFIG_PATH =  current_dir + "streaming_ordering_chatbot/resources/brand_configs.json"
with open(BRAND_CONFIG_PATH, 'r') as f:
    brand_configs = json.load(f)

print(f"Loaded {len(brand_configs)} brand configurations:")
for brand in brand_configs:
    print(f"- {brand_configs[brand]['name']}")


Loaded 1 brand configurations:
- Contoso Restaurant


In [24]:
brand_configs

{'Contoso Restaurant': {'name': 'Contoso Restaurant',
  'tone': 'warm and welcoming',
  'style': 'Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.',
  'key_phrases': ['delicious',
   'home-style',
   'fresh',
   'made with love',
   'organic ingredients'],
  'values': ['quality',
   'community engagement',
   'comfort',
   'customer satisfaction',
   'sustainability']}}

In [ ]:
from azure.ai.evaluation.simulator import Simulator
from azure.ai.evaluation._model_configurations import AzureOpenAIModelConfiguration

if not model_config.get("azure_endpoint"):
	raise ValueError("Missing required Azure endpoint in model_config.")
if not model_config.get("azure_deployment"):
	raise ValueError("Missing required Azure deployment in model_config.")
if not model_config.get("api_key"):
	raise ValueError("Missing required API key in model_config.")

azure_model_config = AzureOpenAIModelConfiguration(
	azure_endpoint=str(model_config["azure_endpoint"]),
	azure_deployment=str(model_config["azure_deployment"]),
	api_key=str(model_config["api_key"]),
	api_version=str(model_config.get("api_version", "2024-02-15-preview"))
)

simulator = Simulator(model_config=azure_model_config)

Class Simulator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [ ]:
from typing import List, Dict, Any, Optional
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider


def call_to_restaurant_chatbot(query: str, brand_name: str) -> str:
    """Call the restaurant chatbot with the given query and brand personality."""
    try:
        if ENDPOINT is None:
            raise ValueError("Azure endpoint (ENDPOINT) is not set. Please check your environment variables or configuration.")
        client = AzureOpenAI(
            api_key=API_KEY,
            api_version="2024-02-15-preview",
            azure_endpoint=ENDPOINT
        )
       
        if not isinstance(brand_configs, dict):
            raise ValueError("brand_configs is not a dictionary.")
        brand_config = brand_configs.get(brand_name)
        if not isinstance(brand_config, dict):
            brand_config = {}

        brand_instructions = f"""You are representing {brand_config.get('name', 'a restaurant')}.
        
        TONE AND STYLE:
        - Tone: {brand_config.get('tone', '')}
        - Style: {brand_config.get('style', '')}
        
        BRAND VOICE GUIDELINES:
        1. Key phrases to naturally incorporate: {', '.join(brand_config.get('key_phrases', []))}
        2. Core brand values to embody: {', '.join(brand_config.get('values', []))}
        
        Maintain this brand voice consistently while helping customers with their orders."""
        
        if DEPLOYMENT_NAME is None:
            raise ValueError("Azure deployment name (DEPLOYMENT_NAME) is not set. Please check your environment variables or configuration.")
        completion = client.chat.completions.create(
            model=str(DEPLOYMENT_NAME),
            messages=[
                {
                    "role": "system",
                    "content": brand_instructions
                },
                {
                    "role": "user",
                    "content": query,
                }
            ],
            max_tokens=800,
            temperature=0.7,
            top_p=0.95,
        )
        message = completion.choices[0].message
        
        return message.content if message.content is not None else ""
    except Exception as e:
        return f"Error calling restaurant chatbot: {e}"


async def callback(
    messages: Dict[str, List[Dict]],
    stream: bool = False,
    session_state: Any = None,
    context: Optional[Dict[str, Any]] = None,
) -> dict:
    """Callback function for the simulator to interact with the chatbot."""
    messages_list = messages["messages"]
    latest_message = messages_list[-1]
    query = latest_message["content"]
    
    # Get brand from context
    if context is not None and isinstance(context, dict):
        brand_name = context.get("brand_name", "")
    else:
        brand_name = "casual"
    
   
    response = call_to_restaurant_chatbot(query, brand_name)
    
    formatted_response = {
        "content": response,
        "role": "assistant",
        "context": {
            "citations": None,
            "brand_name": brand_name
        },
    }
    messages["messages"].append(formatted_response)
    return {"messages": messages["messages"], "stream": stream, "session_state": session_state, "context": context}

In [5]:
import wikipedia

wiki_search_term = "restaurant"
wiki_title = wikipedia.search(wiki_search_term)[0]
wiki_page = wikipedia.page(wiki_title)
text = wiki_page.summary[:5000]

In [ ]:
# Define test scenarios for restaurant ordering
test_scenarios = [
        f"I want to order dinner for my family from your {wiki_search_term}",
        f"What are your most popular dishes in your {wiki_search_term}?",
        f"I'm vegetarian, what options do your {wiki_search_term} serve?",
        f"I'm looking for low-calorie options, do your {wiki_search_term} have any?",
        f"I need catering for an office party of 20 people, can your {wiki_search_term} handle that?",
        f"We're celebrating a birthday, what do your {wiki_search_term} have for 50 guest?"
        f"Can your {wiki_search_term} accept customize order?",
        f"I need extra spicy, do your {wiki_search_term} have spicy options?"
    ]

current_directory = Path.cwd()
prompt_dir = Path(current_directory) / "prompts"

simulator_prompt = Path(prompt_dir) / "restaurant_simulator.prompty"
#with open(simulator_prompt, 'r') as f:
#        doc = f.read()
'''
try:
    import yaml
    with open(simulator_prompt, 'r') as f:
        doc = f.read()
        # Load all YAML documents from the file
        docs = list(yaml.safe_load_all(doc))
        # First document contains the configuration
        simulator_prompty = docs[0]
        # Second document contains the system prompt
        system_prompt = docs[1] if len(docs) > 1 else None
except (yaml.YAMLError, FileNotFoundError) as e:
    print(f"Error reading simulator prompt: {e}")
    simulator_prompty = None
    system_prompt = None
'''
# Verify the prompt file exists
if not simulator_prompt.exists():
    raise FileNotFoundError(f"Prompt file not found at {simulator_prompt}")

In [11]:
doc

'---\nname: TaskSimulatorWithPersona\ndescription: Simulates a restaurant customer with specific persona\nmodel:\n  api: chat\n  parameters:\n    temperature: 0.7\n    top_p: 0.95\n    presence_penalty: 0\n    frequency_penalty: 0\n    response_format:\n      type: json_object\n\ninputs:\n  task:\n    type: string\n  mood:\n    type: string\n  tone:\n    type: string\n  style:\n    type: string\n  values:\n    type: string\n\n\n---\nsystem: \nYou are simulating a customer interacting with a {{ brand }} restaurant to complete a specific task: {{ task }}, you are mood: {{ mood }}.\n\nThe restaurant has this brand personality, with tone: {{ tone }}, style: {{ style }}, and values: {{ values }}.\n\nYou must behave as a user who wants accomplish this task: {{ task }} and you continue to interact with a system that responds to your queries. If there is a message in the conversation history from the assistant, make sure you read the content of the message and include it your first response. Y

In [12]:
user_prompty_kwargs = {
            "brand": str(brand_config["name"]).strip(),
            "mood": str(mood).strip(),
            "tone": str(brand_config["tone"]).strip(),
            "style": str(brand_config["style"]).strip(),
            "values": str(', '.join(brand_config["values"])).strip(),
            "conversation_history": conversation_history,
            "task": test_scenarios[0].strip()  # Use the first scenario as the initial task
            }
user_prompty_kwargs

{'brand': 'Contoso Restaurant',
 'mood': 'happy',
 'tone': 'warm and welcoming',
 'style': 'Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.',
 'values': 'quality, community engagement, comfort, customer satisfaction, sustainability',
 'conversation_history': {'messages': [], 'current_turn': 0},
 'task': 'I want to order dinner for my family from your restaurant'}

In [105]:
brand_config['values']

['quality',
 'community engagement',
 'comfort',
 'customer satisfaction',
 'sustainability']

In [ ]:
# Initialize results list for evaluation
results = []

moods = ["happy", "rushed", "indecisive", "demanding", "curious"]

for brand_name, brand_config in brand_configs.items():
    print(f"\nTesting brand: {brand_config['name']}")        
    for mood in moods:
            conversation_history = {
                "messages": [],  
                "current_turn": 0
            }
            
            user_prompty_kwargs = {
            "brand": str(brand_config["name"]).strip(),
            "mood": str(mood).strip(),
            "tone": str(brand_config["tone"]).strip(),
            "style": str(brand_config["style"]).strip(),
            "values": str(', '.join(brand_config["values"])).strip(),
            "conversation_history": conversation_history,
            "task": test_scenarios[0].strip()  # Use the first scenario as the initial task
            }
            
            # Print debug information
            print(f"\nTesting with configuration:")
            print(f"Mood: {user_prompty_kwargs['mood']}")
            print(f"Brand: {user_prompty_kwargs['brand']}")
            print(f"Tone: {user_prompty_kwargs['tone']}")
            print(f"Style: {user_prompty_kwargs['style']}")
            print(f"Values: {user_prompty_kwargs['values']}")
            print(f"Task: {user_prompty_kwargs['task']}")
                       
            outputs = await simulator(
                        target=callback,
                        text=text,
                        num_queries=7,
                        max_conversation_turns=5,
                        tasks=test_scenarios,
                        user_simulator_prompty=simulator_prompt.absolute().as_posix(),
                        user_simulator_prompty_options=user_prompty_kwargs,
                        context={"brand_name": brand_name}
                    )
                    
                    # Add metadata to results
            for output in outputs:
                output["metadata"] = {
                            "brand_name": brand_name,
                            "brand_config": brand_config,
                            "mood": mood
                        }
                results.append(output)


Testing brand: Contoso Restaurant

Testing with configuration:
Mood: happy
Brand: Contoso Restaurant
Tone: warm and welcoming
Style: Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.
Values: quality, community engagement, comfort, customer satisfaction, sustainability
Task: I want to order dinner for my family from your restaurant






































































































































































































































































































































Generating: 100%|██████████████████████████████████████████████| 35/35 [05:54<00:00, 10.14s/message]



Testing with configuration:
Mood: rushed
Brand: Contoso Restaurant
Tone: warm and welcoming
Style: Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.
Values: quality, community engagement, comfort, customer satisfaction, sustainability
Task: I want to order dinner for my family from your restaurant






































































































































































































































































































































Generating: 100%|██████████████████████████████████████████████| 35/35 [07:13<00:00, 12.38s/message]



Testing with configuration:
Mood: indecisive
Brand: Contoso Restaurant
Tone: warm and welcoming
Style: Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.
Values: quality, community engagement, comfort, customer satisfaction, sustainability
Task: I want to order dinner for my family from your restaurant






































































































































































































































































































































Generating: 100%|██████████████████████████████████████████████| 35/35 [06:19<00:00, 10.85s/message]



Testing with configuration:
Mood: demanding
Brand: Contoso Restaurant
Tone: warm and welcoming
Style: Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.
Values: quality, community engagement, comfort, customer satisfaction, sustainability
Task: I want to order dinner for my family from your restaurant






































































































































































































































































































































Generating: 100%|██████████████████████████████████████████████| 35/35 [05:51<00:00, 10.03s/message]



Testing with configuration:
Mood: curious
Brand: Contoso Restaurant
Tone: warm and welcoming
Style: Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.
Values: quality, community engagement, comfort, customer satisfaction, sustainability
Task: I want to order dinner for my family from your restaurant






































































































































































































































































































































Generating: 100%|██████████████████████████████████████████████| 35/35 [06:11<00:00, 10.60s/message]


In [30]:
from pathlib import Path

# Save results with clear structure
evaluation_results = {
    "metadata": {
        "timestamp": "2025-06-25",
        "num_brands": len(brand_configs),
        #"num_scenarios": sum(len(scenarios) for scenarios in test_scenarios.values()),
        "num_moods": len(moods)
    },
    "results": results
}

output_file = Path("chatbot_evaluation_data.json")
with output_file.open("w") as f:
    json.dump(evaluation_results, f, indent=2)

print(f"\nEvaluation complete! Results saved to {output_file}")
print(f"Tested {len(brand_configs)} brands with {len(moods)} moods.")


Evaluation complete! Results saved to chatbot_evaluation_data.json
Tested 1 brands with 5 moods.


In [ ]:
from azure.ai.evaluation import QAEvaluator
import pandas as pd


with open("chatbot_evaluation_results.json", "r") as f:
    eval_data = json.load(f)


eval_inputs = []
for result in eval_data["results"]:
    conversation = result["conversation"]
    metadata = result["metadata"]
    brand_config = metadata["brand_config"]
    
    # Process each turn in the conversation
    for i in range(0, len(conversation)-1, 2):  # Step by 2 to get user-assistant pairs
        user_message = conversation[i]["content"]
        assistant_message = conversation[i+1]["content"]
        
        eval_input = {
            "query": user_message,
            "response": assistant_message,
            "context": {
                "brand_name": metadata["brand_name"],
                "tone": brand_config["tone"],
                "style": brand_config["style"],
                "values": brand_config["values"],
                "scenario_type": metadata["scenario_type"],
                "mood": metadata["mood"]
            }
        }
        eval_inputs.append(eval_input)


with open("eval_input_data.json", "w") as f:
    json.dump(eval_inputs, f, indent=2)

print(f"Prepared {len(eval_inputs)} conversation turns for evaluation")

In [ ]:
# Initialize QAEvaluator with custom metrics
evaluator = QAEvaluator(
    metrics={
        "brand_voice_consistency": {
            "prompt": """
            Evaluate how well the assistant's response maintains the brand's voice and tone.
            Consider:
            - Does it match the specified tone ({context[tone]})?
            - Does it reflect the brand's style ({context[style]})?
            - Does it embody the brand's values ({context[values]})?
            
            Score from 1-5 where:
            1: Completely mismatched with brand voice
            3: Partially maintains brand voice
            5: Perfect alignment with brand voice
            """,
            "min_score": 1,
            "max_score": 5
        },
        "response_relevance": {
            "prompt": """
            Evaluate how relevant and appropriate the assistant's response is to the user's query.
            Consider:
            - Does it directly address the user's question/request?
            - Is the information accurate and helpful?
            - Is it contextually appropriate for a restaurant ordering scenario?
            
            Score from 1-5 where:
            1: Completely irrelevant or inappropriate
            3: Partially relevant/appropriate
            5: Perfectly relevant and appropriate
            """,
            "min_score": 1,
            "max_score": 5
        },
        "task_completion": {
            "prompt": """
            Evaluate how well the assistant helps progress or complete the restaurant ordering task.
            Consider:
            - Does it move the conversation toward completing the order?
            - Does it handle necessary details (menu items, quantities, modifications)?
            - Does it resolve any issues or questions effectively?
            
            Score from 1-5 where:
            1: No progress toward task completion
            3: Some progress but with gaps
            5: Excellent progress/completion
            """,
            "min_score": 1,
            "max_score": 5
        }
    }
)

print("QAEvaluator configured with custom metrics for brand voice, relevance, and task completion")

In [ ]:
# Run evaluation on prepared data
evaluation_results = evaluator.evaluate(eval_inputs)

# Save detailed results
with open("restaurant_chatbot_evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)

# Create a DataFrame for analysis
results_data = []
for result in evaluation_results:
    metrics = result["metrics"]
    context = result["context"]
    
    row = {
        "brand_name": context["brand_name"],
        "tone": context["tone"],
        "scenario_type": context["scenario_type"],
        "mood": context["mood"],
        "brand_voice_score": metrics["brand_voice_consistency"]["score"],
        "relevance_score": metrics["response_relevance"]["score"],
        "task_completion_score": metrics["task_completion"]["score"],
        "query": result["query"],
        "response": result["response"]
    }
    results_data.append(row)

results_df = pd.DataFrame(results_data)

# Save detailed analysis
results_df.to_csv("restaurant_chatbot_detailed_analysis.csv", index=False)

# Calculate and display summary metrics
summary = results_df.groupby("brand_name").agg({
    "brand_voice_score": ["mean", "std"],
    "relevance_score": ["mean", "std"],
    "task_completion_score": ["mean", "std"]
}).round(2)

print("\nEvaluation Summary by Brand:")
print(summary)

# Create visualizations
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15, 6))

# Plot 1: Brand Voice Consistency
plt.subplot(1, 3, 1)
sns.boxplot(data=results_df, x="brand_name", y="brand_voice_score")
plt.title("Brand Voice Consistency")
plt.xticks(rotation=45)

# Plot 2: Response Relevance
plt.subplot(1, 3, 2)
sns.boxplot(data=results_df, x="brand_name", y="relevance_score")
plt.title("Response Relevance")
plt.xticks(rotation=45)

# Plot 3: Task Completion
plt.subplot(1, 3, 3)
sns.boxplot(data=results_df, x="brand_name", y="task_completion_score")
plt.title("Task Completion")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Display worst-performing interactions for improvement
print("\nSamples of Low-Scoring Interactions:")
for metric in ["brand_voice_score", "relevance_score", "task_completion_score"]:
    print(f"\nLowest {metric}:")
    worst = results_df.nsmallest(3, metric)[["brand_name", metric, "query", "response"]]
    print(worst.to_string())

In [ ]:
from azure.ai.evaluation import QAEvaluator
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# Initialize the evaluator
evaluator = QAEvaluator(model_config=model_config)